# Персональные данные&LLM ИИ ЧЕМП: Альфа-Банк

МОЯ МОДЕЛЬ

люблю кошек из питера

## Шаг 1: Загрузка и препроцессинг данных

In [ ]:
!pip install -q pandas numpy scikit-learn transformers datasets torch

In [ ]:
import ast
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)
from datasets import Dataset
import torch
import warnings

warnings.filterwarnings("ignore")

# Базовая модель DeepPavlov
MODEL_NAME = "DeepPavlov/rubert-base-cased"
MAX_LENGTH = 256
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Загрузка
df = pd.read_csv("/content/train_dataset.tsv", sep="	")
df["target_parsed"] = df["target"].apply(ast.literal_eval)
print(f"Загружено {len(df)} примеров")


Device: cuda
Загружено 8287 примеров


In [ ]:
# Убираем вложенные спаны: если спан A полностью внутри спана B — выкидываем A
def remove_nested_spans(entities):
    if len(entities) < 2:
        return entities
    sorted_ents = sorted(entities, key=lambda x: (x[1] - x[0]), reverse=True)  # длинные первыми
    kept = []
    for ent in sorted_ents:
        s, e, cat = ent
        is_nested = False
        for ks, ke, kcat in kept:
            if ks <= s and ke >= e and (ks, ke) != (s, e):
                is_nested = True
                break
        if not is_nested:
            kept.append(ent)
    return sorted(kept, key=lambda x: x[0])

nested_before = sum(len(ents) for ents in df["target_parsed"])
df["entities"] = df["target_parsed"].apply(remove_nested_spans)
nested_after = sum(len(ents) for ents in df["entities"])
print(f"Сущностей до: {nested_before}, после удаления вложенных: {nested_after}, убрано: {nested_before - nested_after}")

Сущностей до: 7158, после удаления вложенных: 7105, убрано: 53


In [ ]:
# Собираем label2id маппинг
all_categories = sorted(set(cat for ents in df["entities"] for _, _, cat in ents))
print(f"Категорий: {len(all_categories)}")
for c in all_categories:
    print(f"  {c}")

label_list = ["O"]
for cat in all_categories:
    label_list.append(f"B-{cat}")
    label_list.append(f"I-{cat}")

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print(f"\nВсего меток: {len(label_list)}")

Категорий: 30
  API ключи
  CVV/CVC
  Email
  Водительское удостоверение
  Временное удостоверение личности
  Гражданство и названия стран
  Данные об автомобиле клиента
  Данные об организации/юридическом лице (ИНН, КПП, ОГРН, БИК, адреса, расчётный счёт)
  Дата окончания срока действия карты
  Дата регистрации по месту жительства или пребывания
  Дата рождения
  Имя держателя карты
  Кодовые слова
  Место рождения
  Наименование банка
  Номер банковского счета
  Номер карты
  Номер телефона
  Одноразовые коды
  ПИН код
  Пароли
  Паспортные данные
  Полный адрес
  Разрешение на работу / визу
  СНИЛС клиента
  Сведения об ИНН
  Свидетельство о рождении
  Серия и номер вида на жительство
  Содержимое магнитной полосы
  ФИО

Всего меток: 61


In [ ]:
# Train/val split — стратификация по наличию сущностей
df["has_entity"] = df["entities"].apply(lambda x: len(x) > 0).astype(int)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["has_entity"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")
print(f"Train с сущн.: {train_df['has_entity'].mean()*100:.1f}%, Val с сущн.: {val_df['has_entity'].mean()*100:.1f}%")

Train: 6629, Val: 1658
Train с сущн.: 68.8%, Val с сущн.: 68.8%


## Шаг 2: Конвертация span → BIO

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(texts, entities_list):
    """Токенизация + конвертация char-спанов в BIO-метки.

    Каждый токен (включая субтокены) получает метку по char-offset overlap.
    Спецтокены ([CLS], [SEP], [PAD]) → -100.
    """
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_offsets_mapping=True,
    )

    all_labels = []
    for i, (offsets, entities) in enumerate(zip(tokenized["offset_mapping"], entities_list)):
        labels = []
        # Для определения B- vs I-: отслеживаем, какая сущность "открыта"
        prev_ent_idx = None

        for tok_start, tok_end in offsets:
            # Спецтокены — offset (0, 0)
            if tok_start == 0 and tok_end == 0:
                labels.append(-100)
                prev_ent_idx = None
                continue

            # Ищем сущность по overlap
            label = "O"
            matched_ent_idx = None
            for ent_idx, (ent_start, ent_end, cat) in enumerate(entities):
                if tok_start < ent_end and tok_end > ent_start:
                    matched_ent_idx = ent_idx
                    if prev_ent_idx == ent_idx:
                        label = f"I-{cat}"
                    else:
                        label = f"B-{cat}"
                    break

            labels.append(label2id[label])
            prev_ent_idx = matched_ent_idx

        all_labels.append(labels)

    tokenized["labels"] = all_labels
    return tokenized

# Тестируем на реальном примере из данных
test_row = train_df[train_df["entities"].apply(len) > 0].iloc[0]
sample = tokenize_and_align_labels([test_row["text"]], [test_row["entities"]])
tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"][0])
print(f"Текст: {test_row['text'][:100]}...")
print(f"Сущности: {test_row['entities']}\n")
for tok, lab, (ts, te) in zip(tokens, sample["labels"][0], sample["offset_mapping"][0]):
    lab_str = id2label[lab] if lab != -100 else "IGN"
    if lab_str != "O" and lab_str != "IGN":
        print(f"  {tok:20s} offset=({ts:3d},{te:3d}) → {lab_str}")

Invalid model-index. Not loading eval results into CardData.


Текст: Да, с визой 69 5675679 вы можете оформить карту лояльности. Это можно сделать в отделении или онлайн...
Сущности: [(12, 22, 'Разрешение на работу / визу')]

  ▁69                  offset=( 12, 14) → B-Разрешение на работу / визу
  ▁5                   offset=( 15, 16) → I-Разрешение на работу / визу
  675                  offset=( 16, 19) → I-Разрешение на работу / визу
  679                  offset=( 19, 22) → I-Разрешение на работу / визу


In [ ]:
# Валидация: обратная конвертация BIO → спаны, сравнение с оригиналом

STRIP_TRAILING = set(".,;:!)")
STRIP_LEADING = set(" \t\n")

def bio_to_spans(labels_ids, offsets, text=None):
    """Из BIO-меток и offset_mapping восстанавливаем char-спаны.

    Все токены размечены (B/I/O), -100 только у спецтокенов.
    """
    spans = []
    current_cat = None
    current_start = None
    current_end = None

    for lab_id, (tok_start, tok_end) in zip(labels_ids, offsets):
        # Спецтокены
        if lab_id == -100:
            if current_cat is not None:
                spans.append((current_start, current_end, current_cat))
                current_cat = None
            continue

        lab = id2label[lab_id]

        if lab.startswith("B-"):
            if current_cat is not None:
                spans.append((current_start, current_end, current_cat))
            current_cat = lab[2:]
            current_start = tok_start
            current_end = tok_end
        elif lab.startswith("I-") and current_cat == lab[2:]:
            current_end = tok_end
        else:
            if current_cat is not None:
                spans.append((current_start, current_end, current_cat))
                current_cat = None

    if current_cat is not None:
        spans.append((current_start, current_end, current_cat))

    # Постпроцессинг: strip ведущих пробелов и завершающей пунктуации
    if text is not None:
        cleaned = []
        for start, end, cat in spans:
            while start < end and text[start] in STRIP_LEADING:
                start += 1
            while end > start and text[end - 1] in STRIP_TRAILING:
                end -= 1
            if end > start:
                cleaned.append((start, end, cat))
        spans = cleaned

    return spans

# Проверяем round-trip на всём train
texts = train_df["text"].tolist()
entities = train_df["entities"].tolist()
tok_result = tokenize_and_align_labels(texts, entities)

match = 0
total = 0
boundary_mismatch = 0
for i in range(len(texts)):
    offsets = tok_result["offset_mapping"][i]
    predicted_spans = bio_to_spans(tok_result["labels"][i], offsets, text=texts[i])
    gold_spans = [(s, e, c) for s, e, c in entities[i]]
    total += len(gold_spans)
    pred_set = set(predicted_spans)
    for gs in gold_spans:
        if gs in pred_set:
            match += 1
        else:
            for ps in predicted_spans:
                if ps[2] == gs[2] and abs(ps[0] - gs[0]) <= 2 and abs(ps[1] - gs[1]) <= 2:
                    boundary_mismatch += 1
                    break

print(f"Валидация BIO round-trip: {match}/{total} спанов совпали точно ({match/total*100:.1f}%)")
print(f"  Почти совпали (±2 символа): {boundary_mismatch}")
if match < total:
    print(f"  Потеряно полностью: {total - match - boundary_mismatch}")

# Показываем примеры расхождений
shown = 0
for i in range(len(texts)):
    offsets = tok_result["offset_mapping"][i]
    predicted_spans = bio_to_spans(tok_result["labels"][i], offsets, text=texts[i])
    gold_spans = [(s, e, c) for s, e, c in entities[i]]
    pred_set = set(predicted_spans)
    for gs in gold_spans:
        if gs not in pred_set and shown < 5:
            close = [ps for ps in predicted_spans if ps[2] == gs[2]]
            print(f"\n  Gold: {gs} → '{texts[i][gs[0]:gs[1]]}'")
            if close:
                print(f"  Pred: {close[0]} → '{texts[i][close[0][0]:close[0][1]]}'")
            else:
                print(f"  Pred: НЕТ")
            shown += 1

Валидация BIO round-trip: 5680/5734 спанов совпали точно (99.1%)
  Почти совпали (±2 символа): 52
  Потеряно полностью: 2

  Gold: (35, 53, 'Полный адрес') → 'Ленинградская обл.'
  Pred: (27, 33, 'Полный адрес') → 'Россия'

  Gold: (90, 105, 'Полный адрес') → '15-я линия В.О.'
  Pred: (49, 64, 'Полный адрес') → 'Санкт-Петербург'

  Gold: (79, 97, 'Полный адрес') → 'Нижегородская обл.'
  Pred: (71, 77, 'Полный адрес') → 'Россия'

  Gold: (66, 81, 'Полный адрес') → 'Московская обл.'
  Pred: (58, 64, 'Полный адрес') → 'Россия'

  Gold: (78, 93, 'Полный адрес') → 'Московская обл.'
  Pred: (70, 76, 'Полный адрес') → 'Россия'


## Шаг 3: HuggingFace Dataset

In [ ]:
def prepare_dataset(dataframe):
    texts = dataframe["text"].tolist()
    entities = dataframe["entities"].tolist()
    tok = tokenize_and_align_labels(texts, entities)
    return Dataset.from_dict({
        "input_ids": tok["input_ids"],
        "attention_mask": tok["attention_mask"],
        "labels": tok["labels"],
    })

train_dataset = prepare_dataset(train_df)
val_dataset = prepare_dataset(val_df)
print(f"Train dataset: {len(train_dataset)}, Val dataset: {len(val_dataset)}")

Train dataset: 6629, Val dataset: 1658


## Шаг 4–5: Модель и обучение

In [ ]:
!pip install -q seqeval

In [ ]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

# Загружаем модель — classifier head будет заменён на новый (num_labels отличается от оригинала)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
print(f"Модель загружена: {MODEL_NAME}, меток: {len(label2id)}")
print(f"Параметров: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: scanpatch/pii-ner-nemotron
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([55]) vs model:torch.Size([61])            
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([55, 1024]) vs model:torch.Size([61, 1024])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Модель загружена: scanpatch/pii-ner-nemotron, меток: 61
Параметров: 558.9M


In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)

    true_labels = []
    pred_labels = []

    for pred_seq, label_seq in zip(preds, labels):
        true_seq = []
        pred_seq_filtered = []
        for p, l in zip(pred_seq, label_seq):
            if l == -100:
                continue
            true_seq.append(id2label[l])
            pred_seq_filtered.append(id2label[p])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq_filtered)

    return {
        "precision": precision_score(true_labels, pred_labels),
        "recall": recall_score(true_labels, pred_labels),
        "f1": f1_score(true_labels, pred_labels),
    }

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./ner_model_rubert",
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=8,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="best",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Начинаем обучение...")
trainer.train()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Начинаем обучение...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.103330,0.087100,0.758510,0.862044,0.806970
2,0.039769,0.059036,0.848689,0.921168,0.883444
3,0.041262,0.055006,0.832206,0.897810,0.863764
4,0.058304,0.043716,0.894585,0.904380,0.899456
5,0.039916,0.035948,0.888810,0.910219,0.899387
6,0.012072,0.025973,0.876703,0.939416,0.906977
7,0.002403,0.025906,0.921652,0.944526,0.932949
8,0.003039,0.017696,0.957122,0.961314,0.959213
9,0.001886,0.015984,0.957308,0.965693,0.961483
10,0.009735,0.016407,0.964338,0.967153,0.965743


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=16580, training_loss=0.06617065768417048, metrics={'train_runtime': 7116.0017, 'train_samples_per_second': 9.316, 'train_steps_per_second': 2.33, 'total_flos': 4875785370356082.0, 'train_loss': 0.06617065768417048, 'epoch': 10.0})

## Шаг 7: Инференс и финальная оценка на val

In [ ]:
# Посимвольная оценка strict span + category match (micro F1)
val_texts = val_df["text"].tolist()
val_entities = val_df["entities"].tolist()

# Токенизация с offset_mapping для обратной конвертации
val_tok = tokenizer(
    val_texts, truncation=True, max_length=MAX_LENGTH,
    padding=True, return_offsets_mapping=True, return_tensors="pt"
)
offsets_all = val_tok.pop("offset_mapping")

device = next(model.parameters()).device
val_tok = {k: v.to(device) for k, v in val_tok.items()}

model.eval()
with torch.no_grad():
    batch_size = 128
    all_preds = []
    for start in range(0, len(val_texts), batch_size):
        end = min(start + batch_size, len(val_texts))
        batch = {k: v[start:end] for k, v in val_tok.items()}
        logits = model(**batch).logits
        all_preds.append(torch.argmax(logits, dim=-1).cpu().numpy())
    preds = np.concatenate(all_preds, axis=0)

tp = 0
fp = 0
fn = 0

pred_spans_all = []
for i in range(len(val_texts)):
    offsets = offsets_all[i].tolist()
    pred_labels = preds[i].tolist()

    # Маскируем паддинг-токены (offset (0,0) после первого)
    masked_labels = []
    for j, (ts, te) in enumerate(offsets):
        if ts == 0 and te == 0:
            masked_labels.append(-100)
        else:
            masked_labels.append(pred_labels[j])
    # Первый токен [CLS] тоже спец
    masked_labels[0] = -100

    pred_spans = bio_to_spans(masked_labels, offsets, text=val_texts[i])
    gold_spans = set((s, e, c) for s, e, c in val_entities[i])
    pred_spans_set = set((s, e, c) for s, e, c in pred_spans)
    pred_spans_all.append(pred_spans)

    tp += len(gold_spans & pred_spans_set)
    fp += len(pred_spans_set - gold_spans)
    fn += len(gold_spans - pred_spans_set)

precision = tp / (tp + fp) if tp + fp > 0 else 0
recall = tp / (tp + fn) if tp + fn > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

print(f"=== Strict span+category match (micro) ===")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")
print(f"TP={tp}, FP={fp}, FN={fn}")

=== Strict span+category match (micro) ===
Precision: 0.9614
Recall:    0.9621
F1:        0.9617
TP=1319, FP=53, FN=52


## Шаг 8: Per-category F1 и примеры ошибок

In [ ]:
# Per-category метрики
from collections import defaultdict
cat_stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

for i in range(len(val_texts)):
    gold = set((s, e, c) for s, e, c in val_entities[i])
    pred = set((s, e, c) for s, e, c in pred_spans_all[i])

    for s, e, c in gold & pred:
        cat_stats[c]["tp"] += 1
    for s, e, c in pred - gold:
        cat_stats[c]["fp"] += 1
    for s, e, c in gold - pred:
        cat_stats[c]["fn"] += 1

print(f"{'Категория':<35} {'Prec':>6} {'Rec':>6} {'F1':>6} {'Sup':>5}")
print("-" * 65)
for cat in sorted(cat_stats.keys()):
    s = cat_stats[cat]
    p = s["tp"] / (s["tp"] + s["fp"]) if s["tp"] + s["fp"] > 0 else 0
    r = s["tp"] / (s["tp"] + s["fn"]) if s["tp"] + s["fn"] > 0 else 0
    f = 2 * p * r / (p + r) if p + r > 0 else 0
    sup = s["tp"] + s["fn"]
    print(f"{cat:<35} {p:>6.3f} {r:>6.3f} {f:>6.3f} {sup:>5}")

Категория                             Prec    Rec     F1   Sup
-----------------------------------------------------------------
API ключи                            1.000  1.000  1.000    50
CVV/CVC                              1.000  1.000  1.000    41
Email                                0.966  1.000  0.982    28
Водительское удостоверение           0.964  1.000  0.981    53
Временное удостоверение личности     1.000  1.000  1.000    28
Гражданство и названия стран         0.967  0.983  0.975    59
Данные об автомобиле клиента         0.983  0.919  0.950    62
Данные об организации/юридическом лице (ИНН, КПП, ОГРН, БИК, адреса, расчётный счёт)  0.982  0.857  0.915    63
Дата окончания срока действия карты  0.971  0.985  0.978    68
Дата регистрации по месту жительства или пребывания  0.757  0.757  0.757    37
Дата рождения                        0.839  0.820  0.830    89
Имя держателя карты                  0.974  1.000  0.987    37
Кодовые слова                        1.000  1.000 

In [ ]:
# Примеры ошибок
print("=== Примеры ошибок ===\n")
errors_shown = 0
for i in range(len(val_texts)):
    gold = set((s, e, c) for s, e, c in val_entities[i])
    pred = set((s, e, c) for s, e, c in pred_spans_all[i])

    if gold != pred and errors_shown < 10:
        print(f"Текст: {val_texts[i][:120]}...")
        missed = gold - pred
        extra = pred - gold
        if missed:
            for s, e, c in missed:
                print(f"  FN: '{val_texts[i][s:e]}' [{c}] ({s}:{e})")
        if extra:
            for s, e, c in extra:
                print(f"  FP: '{val_texts[i][s:e]}' [{c}] ({s}:{e})")
        print()
        errors_shown += 1

=== Примеры ошибок ===

Текст: Я переехал, и теперь моя регистрация с 03.03.2023. Как мне сообщить об этом в банк?...
  FN: 'с 03.03.2023' [Дата регистрации по месту жительства или пребывания] (37:49)
  FP: '03.03.2023' [Дата регистрации по месту жительства или пребывания] (39:49)

Текст: Не могу найти письмо с договором, которое вы отправляли на договор@bankmail.ru....
  FP: 'договор@bankmail.ru' [Email] (59:78)

Текст: Да, если вы не меняли адрес прописки с 12.12.2017, то эта информация является актуальной....
  FN: '12.12.2017' [Дата регистрации по месту жительства или пребывания] (39:49)
  FP: 'с 12.12.2017' [Дата регистрации по месту жительства или пребывания] (37:49)

Текст: Заказал новую карту, но на ней срок действия до 06/27. А старая была до июня 2027 года. Это нормально?...
  FN: 'июня 2027 года' [Дата окончания срока действия карты] (72:86)

Текст: Мне нужна справка о владении Chery Tiggo 7 Pro 2023 года для налоговой. Как её получить максимально быстро?...
  FN: 'Tiggo 7 P

In [ ]:
import re

# === Регулярки для уточнения границ структурированных данных ===

REGEX_PATTERNS = {
    "Email": r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+',
    "СНИЛС клиента": r'\d{3}[-\s]?\d{3}[-\s]?\d{3}[-\s]?\d{2}',
    "Паспортные данные": r'\d{2}\s?\d{2}\s?\d{6}',
    "Сведения об ИНН": r'\b\d{10}(?:\d{2})?\b',
    "Номер телефона": r'(?:\+7|8|7)[\s\-]?\(?\d{3}\)?[\s\-]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}',
    "Номер карты": r'\d{4}[\s\-]?\d{4}[\s\-]?\d{4}[\s\-]?\d{4}',
    "Номер банковского счета": r'\d{20}',
    "CVV/CVC": r'\b\d{3}\b',
}


def refine_exact_match(text, start, end, category):
    """Уточняет границы структурированных данных регуляркой в окне вокруг предсказания модели."""
    if category not in REGEX_PATTERNS:
        return start, end

    window_start = max(0, start - 5)
    window_end = min(len(text), end + 5)
    window = text[window_start:window_end]

    for match in re.finditer(REGEX_PATTERNS[category], window):
        new_start = window_start + match.start()
        new_end = window_start + match.end()
        # Берём совпадение только если оно пересекается с исходным спаном
        if new_start < end and new_end > start:
            return new_start, new_end

    return start, end


# === Blacklist корпоративных email'ов ===

BLACKLIST_DOMAINS = [
    '@bankmail.ru', '@alfabank.ru', '@alfa.ru',
]
BLACKLIST_PREFIXES = [
    'info@', 'support@', 'dogovor@', 'noreply@', 'test@',
]


def is_corporate_email(email_text):
    email_lower = email_text.lower()
    for domain in BLACKLIST_DOMAINS:
        if domain in email_lower:
            return True
    for prefix in BLACKLIST_PREFIXES:
        if email_lower.startswith(prefix):
            return True
    return False


# === Лечение "съеденных" точек в адресах ===

ADDRESS_CATEGORIES = {"Полный адрес"}
ADDRESS_ABBREVS = ('обл', 'г', 'ул', 'д', 'кв', 'пер', 'просп', 'ш', 'р-н', 'бул', 'пр', 'стр', 'корп')


def fix_address_punctuation(text, start, end, category):
    if category in ADDRESS_CATEGORIES:
        if end < len(text) and text[end] == '.' and text[start:end].endswith(ADDRESS_ABBREVS):
            end += 1
    return start, end


# === Расширение границ для наименований банков ===

def fix_bank_names(text, start, end, category):
    if category == "Наименование банка":
        window_start = max(0, start - 10)
        prefix_text = text[window_start:start]
        match = re.search(r'(?i)\b(банк[ауеом]?|пао|ао)\s+$', prefix_text)
        if match:
            start = window_start + match.start()
    return start, end


# === Удаление предлогов из начала спана ===

PREPOSITION_RE = re.compile(r'^(?:с|до|по|в|от|на|за|из|для|без|при|через|после)\s+', re.IGNORECASE)


def strip_leading_prepositions(text, start, end, category):
    """Убирает предлоги из начала любого спана."""
    span_text = text[start:end]
    m = PREPOSITION_RE.match(span_text)
    if m:
        start += m.end()
    return start, end


# === Постобработка: применяем все правила ===

def postprocess_spans(text, raw_spans):
    final = []
    for start, end, category in raw_spans:
        # 1. Точки в адресах
        start, end = fix_address_punctuation(text, start, end, category)
        # 2. Расширяем банки
        # start, end = fix_bank_names(text, start, end, category)
        # # 3. Убираем предлоги
        # start, end = strip_leading_prepositions(text, start, end, category)
        # # 4. Регулярки для структурированных данных
        # start, end = refine_exact_match(text, start, end, category)
        # # 5. Blacklist email
        # if category == "Email":
        #     if is_corporate_email(text[start:end]):
        #         continue
        # # Добавляем, если валидный спан
        if start < end:
            final.append((start, end, category))
    return final


print("Regex-постобработка определена")

Regex-постобработка определена


In [ ]:
def run_inference(model, tokenizer, texts, batch_size=128):
    """Батчевый инференс, возвращает raw-спаны (до постобработки)."""
    model.eval()
    device = next(model.parameters()).device
    all_spans = []

    for batch_start in range(0, len(texts), batch_size):
        batch_end = min(batch_start + batch_size, len(texts))
        batch_texts = texts[batch_start:batch_end]

        tok = tokenizer(
            batch_texts, truncation=True, max_length=MAX_LENGTH,
            padding=True, return_offsets_mapping=True, return_tensors="pt"
        )
        offsets_batch = tok.pop("offset_mapping")
        tok = {k: v.to(device) for k, v in tok.items()}

        with torch.no_grad():
            logits = model(**tok).logits
        preds_batch = torch.argmax(logits, dim=-1).cpu().numpy()

        for i, text in enumerate(batch_texts):
            offsets = offsets_batch[i].tolist()
            pred_labels = preds_batch[i].tolist()

            # Маскируем спецтокены (offset (0,0))
            masked_labels = []
            for j, (ts, te) in enumerate(offsets):
                if ts == 0 and te == 0:
                    masked_labels.append(-100)
                else:
                    masked_labels.append(pred_labels[j])
            masked_labels[0] = -100

            spans = bio_to_spans(masked_labels, offsets, text=text)
            all_spans.append(spans)

    return all_spans


def calc_f1(texts, pred_spans_all, gold_entities):
    """Считает strict span+category micro F1."""
    tp = fp = fn = 0
    for i in range(len(texts)):
        gold = set((s, e, c) for s, e, c in gold_entities[i])
        pred = set((s, e, c) for s, e, c in pred_spans_all[i])
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
    return precision, recall, f1, tp, fp, fn


# Инференс
val_texts = val_df["text"].tolist()
val_entities = val_df["entities"].tolist()

raw_spans = run_inference(model, tokenizer, val_texts)

# Без regex
p, r, f1, tp, fp, fn = calc_f1(val_texts, raw_spans, val_entities)
print(f"=== БЕЗ regex постобработки ===")
print(f"Precision: {p:.4f}  Recall: {r:.4f}  F1: {f1:.4f}")
print(f"TP={tp}, FP={fp}, FN={fn}")

# С regex
postprocessed_spans = [postprocess_spans(text, spans) for text, spans in zip(val_texts, raw_spans)]

p2, r2, f12, tp2, fp2, fn2 = calc_f1(val_texts, postprocessed_spans, val_entities)
print(f"\n=== С regex постобработкой ===")
print(f"Precision: {p2:.4f}  Recall: {r2:.4f}  F1: {f12:.4f}")
print(f"TP={tp2}, FP={fp2}, FN={fn2}")

print(f"\nДельта F1: {(f12 - f1)*100:+.2f}%")

=== БЕЗ regex постобработки ===
Precision: 0.9614  Recall: 0.9621  F1: 0.9617
TP=1319, FP=53, FN=52

=== С regex постобработкой ===
Precision: 0.9643  Recall: 0.9650  F1: 0.9646
TP=1323, FP=49, FN=48

Дельта F1: +0.29%


## Шаг 9: Предсказание на тестовом датасете

In [ ]:
# Загрузка тестового датасета
test_df = pd.read_csv("/content/private_test_dataset.csv")
test_texts = test_df["text"].tolist()
print(f"Тестовых примеров: {len(test_texts)}")
print(f"Пример: {test_texts[1][:100]}...")

# Инференс
raw_test_spans = run_inference(model, tokenizer, test_texts)

# Постобработка regex
all_test_spans = [postprocess_spans(text, spans) for text, spans in zip(test_texts, raw_test_spans)]

# Формируем submission
test_df["Prediction"] = [
    str([(s, e, c) for s, e, c in spans]) if spans else "[]"
    for spans in all_test_spans
]

# Статистика
n_with = sum(1 for spans in all_test_spans if spans)
n_total_ents = sum(len(spans) for spans in all_test_spans)
print(f"\nТекстов с сущностями: {n_with}/{len(test_texts)} ({n_with/len(test_texts)*100:.1f}%)")
print(f"Всего найдено сущностей: {n_total_ents}")

# Сохраняем
test_df[["id", "Prediction"]].to_csv("submission.csv", index=False)
print(f"\nСохранено в submission.csv")
test_df[["id", "Prediction"]].head(10)

Тестовых примеров: 3552
Пример: Уточните, пожалуйста, ФИО получателя и точную сумму перевода. Мы проверим информацию по операции на ...

Текстов с сущностями: 2469/3552 (69.5%)
Всего найдено сущностей: 3118

Сохранено в submission.csv


,id,Prediction
0,0,[]
1,1,"[(106, 124, 'Номер телефона')]"
2,2,[]
3,3,"[(88, 93, 'Паспортные данные'), (102, 108, 'Па..."
4,4,"[(82, 94, 'Водительское удостоверение')]"
5,5,"[(51, 63, 'Временное удостоверение личности')]"
6,6,[]
7,7,"[(10, 24, 'СНИЛС клиента')]"
8,8,"[(57, 61, 'ПИН код')]"
9,9,"[(101, 155, 'Содержимое магнитной полосы')]"


In [ ]:
SAVE_DIR = "./ner_model_rubert/best"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Модель и токенизатор сохранены в {SAVE_DIR}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель и токенизатор сохранены в ./ner_model_nemotron/best
